## Objective
Integrate the customer query classification and RAG
components into a unified REST API.

In [ ]:
!pip -q install fastapi uvicorn python-multipart \
    nest-asyncio httpx pymupdf faiss-cpu \
    sentence-transformers transformers sentencepiece accelerate

## previously trained models

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded:
    print(filename)

Saving chunks.json to chunks.json
Saving customer_query_classifier.pkl to customer_query_classifier.pkl
Saving index.faiss to index.faiss
Saving tfidf_vectorizer.pkl to tfidf_vectorizer.pkl
Uploaded files:
chunks.json
customer_query_classifier.pkl
index.faiss
tfidf_vectorizer.pkl


In [3]:

import os
import json
import joblib
import faiss
import torch
import re
import fitz
import numpy as np

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

# Load classification models
tfidf = joblib.load("/content/tfidf_vectorizer.pkl")

classifier = joblib.load("/content/customer_query_classifier.pkl")

print("Classification models loaded successfully!")


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Classification models loaded successfully!


## Load FAISS and document chunks

In [4]:

# Load FAISS index
index = faiss.read_index("index.faiss")

# Load document chunks
with open(
    "chunks.json",
    "r",
    encoding="utf-8"
) as f:

    all_chunks = json.load(f)

print("FAISS index loaded:", index.ntotal)
print("Document chunks loaded:", len(all_chunks))

# Validate
assert index.ntotal == len(all_chunks), (
    "Mismatch between FAISS vectors and document chunks!"
)

print("FAISS and document metadata are aligned!")


FAISS index loaded: 27
Document chunks loaded: 27
FAISS and document metadata are aligned!


In [5]:

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


## Load FLAN-T5

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

print("FLAN-T5 loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 loaded successfully!


In [12]:
print("========== MODEL STATUS ==========")

print("TF-IDF loaded:", tfidf is not None)
print("Classifier loaded:", classifier is not None)
print("FAISS vectors:", index.ntotal)
print("Document chunks:", len(all_chunks))
print("Embedding dimension:", index.d)
print("Embedding model dimension:", embedding_model.get_sentence_embedding_dimension())
print("LLM loaded:", llm_model is not None)

print("==================================")

========== MODEL STATUS ==========
TF-IDF loaded: True
Classifier loaded: True
FAISS vectors: 27
Document chunks: 27
Embedding dimension: 384
Embedding model dimension: 384
LLM loaded: True


/tmp/ipykernel_1004/2456805007.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding model dimension:", embedding_model.get_sentence_embedding_dimension())


In [19]:
!pip install -q fastapi uvicorn python-multipart httpx

In [22]:

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
from typing import Optional
import fitz
import re
import numpy as np
import os

app = FastAPI(
    title="NorthStar Customer Support API",
    description="AI-powered customer support using classification and RAG",
    version="1.0.0"
)


# -----------------------------
# REQUEST MODELS
# -----------------------------

class QueryRequest(BaseModel):
    query: str


# -----------------------------
# TEXT EXTRACTION
# -----------------------------

def extract_uploaded_text(file_bytes, filename):

    pages = []

    if filename.lower().endswith(".pdf"):
        pdf = fitz.open(stream=file_bytes, filetype="pdf")

        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text()

            if text.strip():
                pages.append({
                    "page": page_number,
                    "text": text
                })

        pdf.close()

    elif filename.lower().endswith(".txt"):
        text = file_bytes.decode("utf-8", errors="ignore")

        pages.append({
            "page": None,
            "text": text
        })

    else:
        raise HTTPException(
            status_code=400,
            detail="Only PDF and TXT files are supported."
        )

    return pages


# -----------------------------
# TEXT CLEANING AND CHUNKING
# -----------------------------

def clean_uploaded_text(text):

    text = re.sub(r"\s+", " ", text)
    return text.strip()


def create_uploaded_chunks(pages, filename):

    new_chunks = []

    chunk_size = 500
    overlap = 100

    for page_data in pages:

        text = clean_uploaded_text(page_data["text"])

        if not text:
            continue

        start = 0

        while start < len(text):

            end = start + chunk_size
            chunk_text = text[start:end].strip()

            if chunk_text:

                new_chunks.append({
                    "source": filename,
                    "page": page_data["page"],
                    "section": "Uploaded document",
                    "text": chunk_text,
                    "embed_text": filename + ". " + chunk_text,
                    "chunk_id": None
                })

            start += chunk_size - overlap

    return new_chunks


# -----------------------------
# 1. ROOT ENDPOINT
# -----------------------------

@app.get("/")
def home():

    return {
        "message": "NorthStar Customer Support API is running",
        "version": "1.0.0",
        "docs": "/docs"
    }


# -----------------------------
# 2. HEALTH CHECK
# -----------------------------

@app.get("/health")
def health_check():

    return {
        "status": "healthy",
        "classifier_loaded": classifier is not None,
        "vectorizer_loaded": tfidf is not None,
        "faiss_vectors": index.ntotal,
        "document_chunks": len(all_chunks),
        "embedding_model_loaded": embedding_model is not None,
        "llm_loaded": llm_model is not None
    }


# -----------------------------
# 3. QUERY CLASSIFICATION
# -----------------------------

@app.post("/classify-query")
def classify_query(request: QueryRequest):

    query = request.query.strip()

    if not query:
        raise HTTPException(
            status_code=400,
            detail="Query cannot be empty."
        )

    query_vector = tfidf.transform([query])

    prediction = classifier.predict(query_vector)[0]

    return {
        "query": query,
        "predicted_category": str(prediction)
    }


# -----------------------------
# 4. AI RESPONSE GENERATION
# -----------------------------

@app.post("/generate-response")
def generate_response(request: QueryRequest):

    query = request.query.strip()

    if not query:
        raise HTTPException(
            status_code=400,
            detail="Query cannot be empty."
        )

    # Predict customer query category
    query_vector = tfidf.transform([query])
    category = classifier.predict(query_vector)[0]

    # Generate answer using RAG
    result = generate_rag_response(query)

    return {
        "query": query,
        "predicted_category": str(category),
        "answer": result["answer"],
        "status": result["status"],
        "sources": result["sources"]
    }


# -----------------------------
# 5. DOCUMENT UPLOAD
# -----------------------------

@app.post("/upload-document")
async def upload_document(file: UploadFile = File(...)):

    global index, all_chunks

    filename = file.filename or ""

    if not filename.lower().endswith((".pdf", ".txt")):
        raise HTTPException(
            status_code=400,
            detail="Only PDF and TXT files are supported."
        )

    file_bytes = await file.read()

    if not file_bytes:
        raise HTTPException(
            status_code=400,
            detail="Uploaded file is empty."
        )

    # Extract document text
    pages = extract_uploaded_text(file_bytes, filename)

    # Create chunks
    new_chunks = create_uploaded_chunks(pages, filename)

    if not new_chunks:
        raise HTTPException(
            status_code=400,
            detail="No readable text found in the uploaded document."
        )

    # Generate embeddings
    texts = [chunk["embed_text"] for chunk in new_chunks]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Assign unique chunk IDs
    starting_id = len(all_chunks)

    for i, chunk in enumerate(new_chunks):
        chunk["chunk_id"] = starting_id + i

    # Add embeddings to FAISS
    index.add(embeddings)

    # Add metadata
    all_chunks.extend(new_chunks)

    return {
        "message": "Document uploaded and indexed successfully.",
        "filename": filename,
        "pages_processed": len(pages),
        "chunks_added": len(new_chunks),
        "total_vectors": index.ntotal,
        "total_chunks": len(all_chunks)
    }


## Test API

In [23]:

from fastapi.testclient import TestClient

client = TestClient(app)

# Test 1: Root
response = client.get("/")
print("ROOT:", response.json())


# Test 2: Health check
response = client.get("/health")
print("\nHEALTH:", response.json())


# Test 3: Classification
response = client.post(
    "/classify-query",
    json={"query": "Where is my order?"}
)
print("\nCLASSIFICATION:", response.json())


# Test 4: AI response
response = client.post(
    "/generate-response",
    json={"query": "What are the specifications of NS Aero 14?"}
)
print("\nAI RESPONSE:", response.json())


ROOT: {'message': 'NorthStar Customer Support API is running', 'version': '1.0.0', 'docs': '/docs'}

HEALTH: {'status': 'healthy', 'classifier_loaded': True, 'vectorizer_loaded': True, 'faiss_vectors': 27, 'document_chunks': 27, 'embedding_model_loaded': True, 'llm_loaded': True}

CLASSIFICATION: {'query': 'Where is my order?', 'predicted_category': 'Order Status'}

AI RESPONSE: {'query': 'What are the specifications of NS Aero 14?', 'predicted_category': 'Product Inquiry', 'answer': 'Display: 14-inch Full HD display. Memory and storage: 16 GB RAM and 512 GB SSD. Connectivity: Wi-Fi 6 and Bluetooth 5.2. Ports: two USB-A ports, one USB-C port, and an HDMI port. Illustrative price: 54,999. Box contents: laptop, power adapter, and quick-start guide.', 'status': 'answered', 'sources': [{'document': 'product_catalog.pdf', 'page': 1, 'section': 'Laptop – NS Aero 14', 'similarity': 0.6064}, {'document': 'product_catalog.pdf', 'page': 1, 'section': 'Smartphone – NS Nova 5G', 'similarity': 0.44

In [25]:
import threading
import uvicorn
from google.colab.output import eval_js

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

url = eval_js("google.colab.kernel.proxyPort(8000)")
print("API Documentation:", url + "/docs")

INFO:     Started server process [1004]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


API Documentation: https://8000-m-s-euw4b0-13x6kr1ss9fz7-b.europe-west4.prod.r.colab.dev/docs


In [28]:
import requests

try:
    response = requests.get("http://127.0.0.1:8000/", timeout=5)
    print("Status:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("Server is not responding:", e)

INFO:     127.0.0.1:49794 - "GET / HTTP/1.1" 200 OK
Status: 200
Response: {"message":"NorthStar Customer Support API is running","version":"1.0.0","docs":"/docs"}


In [29]:
from google.colab.output import eval_js

print(eval_js("google.colab.kernel.proxyPort(8000)"))

https://8000-m-s-euw4b0-13x6kr1ss9fz7-b.europe-west4.prod.r.colab.dev
